# Indian Deepfake Voice Dataset - Build Pipeline

This notebook downloads, processes and packages the **Indian Deepfake Voice** dataset.

**Rule for this notebook:** all downloading and processing happens in `/content` (Colab local disk), NOT in Google Drive. Google Drive is slow for many small files and has API rate limits. Drive is only used at the very end, to save the final `.tar.gz` backup.

**Final dataset folder name:** `indian-deepfake-voice`


## Step 0: Setup

Install/import libraries and set the folder paths we will use everywhere in this notebook.
Keeping paths in one place means we don't repeat the same string in many cells.


In [ ]:
# Core libraries used across the notebook
import os
import re
import glob
import shutil
from collections import defaultdict

import pandas as pd
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Central path config (edit here if a path ever needs to change)
# ------------------------------------------------------------

# Root of the final, cleaned dataset (lives on local Colab disk, not Drive)
DATASET_ROOT = "/content/indian-deepfake-voice"
FAKE_DIR = os.path.join(DATASET_ROOT, "fake")
REAL_DIR = os.path.join(DATASET_ROOT, "real")

# Raw downloads from Hugging Face (temporary, deleted after processing)
INDIC_SYNTH_RAW_DIR = "/content/IndicSynth"
INDIC_VOICES_RAW_DIR = "/content/IndicVoices"

# Where the final archive backup goes on Google Drive
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/DATASETS"

os.makedirs(FAKE_DIR, exist_ok=True)
os.makedirs(REAL_DIR, exist_ok=True)

print("Dataset root :", DATASET_ROOT)
print("Fake folder  :", FAKE_DIR)
print("Real folder  :", REAL_DIR)

## Step 1: Download Raw Data from Hugging Face

Two datasets come from Hugging Face:

1. **IndicSynth** -> synthetic (fake) Indian language speech
2. **IndicVoices** -> real Indian language speech

Both are downloaded straight into `/content` (local disk), a few files per language, so we don't pull the whole (very large) dataset.


### 1.1 Download IndicSynth (fake voices, source parquet files)

In [ ]:
# NOTE (kept for future reference):
# This is a simpler, single-folder download using snapshot_download.
# Useful when you only need ONE language and want the whole folder at once.
#
# from huggingface_hub import snapshot_download
#
# snapshot_download(
#     repo_id="vdivyasharma/IndicSynth",
#     repo_type="dataset",
#     allow_patterns=["Bengali/*.parquet"],   # only this folder
#     local_dir="IndicSynth_partial"
# )

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

NUM_FILE_SELECT_EACH_CLASS = 4

repo_id = "vdivyasharma/IndicSynth"
api = HfApi()

# Step 1: list every file in the dataset repo
files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")

# Step 2: group the parquet files by language/class (folder name)
files_by_class = defaultdict(list)
for file in files:
    if file.endswith(".parquet"):
        directory = "/".join(file.split("/")[:-1])
        files_by_class[directory].append(file)

# Step 3: pick a few files from each language so the download stays small
selected_files = []
for directory, class_files in files_by_class.items():
    selected = class_files[:NUM_FILE_SELECT_EACH_CLASS]
    selected_files.extend(selected)

    print(f"\n{'=' * 50}")
    print(f"Directory/Class: {directory}")
    print(f"Total files available: {len(class_files)}")
    print(f"Files selected: {len(selected)}")
    for file in selected:
        print(file)

# Step 4: download the selected files into local Colab disk
downloaded_files = []
for filename in selected_files:
    print(f"\nDownloading: {filename}")
    file_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=filename,
        local_dir=INDIC_SYNTH_RAW_DIR
    )
    downloaded_files.append(file_path)

print(f"\n{'=' * 50}")
print("Download completed!")
print(f"Total files downloaded: {len(downloaded_files)}")

In [ ]:
# Delete raw IndicSynth download if you need to start over / free space
# !rm -rf "{INDIC_SYNTH_RAW_DIR}"

### 1.2 Hugging Face login (only needed for gated/private datasets)

In [ ]:
from huggingface_hub import login, logout

# login()   # uncomment and run this if a dataset needs authentication
logout()

### 1.3 Download IndicVoices (real voices, source parquet files)

In [ ]:
FILTER_LANGUAGE = [
    'Bengali', 'Urdu', 'Kannada', 'Marathi', 'Malayalam', 'Tamil',
    'Gujarati', 'Odia', 'Punjabi', 'Telugu', 'Hindi', 'Sanskrit'
]

repo_id = "ai4bharat/IndicVoices"
api = HfApi()

# Step 1: list every file in the dataset repo
files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")

# Step 2: group the parquet files by language/class (folder name)
files_by_class = defaultdict(list)
for file in files:
    if file.endswith(".parquet"):
        directory = "/".join(file.split("/")[:-1])
        files_by_class[directory].append(file)

# Step 3: keep only the Indian languages we want, and cap files per language
selected_files = []
for directory, class_files in files_by_class.items():
    if directory.capitalize() in FILTER_LANGUAGE:
        selected = class_files[:NUM_FILE_SELECT_EACH_CLASS]
        selected_files.extend(selected)

        print(f"\n{'=' * 50}")
        print(f"Directory/Class: {directory}")
        print(f"Total files available: {len(class_files)}")
        print(f"Files selected: {len(selected)}")
        for file in selected:
            print(file)

# Step 4: download the selected files into local Colab disk
downloaded_files = []
for filename in selected_files:
    print(f"\nDownloading: {filename}")
    file_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=filename,
        local_dir=INDIC_VOICES_RAW_DIR
    )
    downloaded_files.append(file_path)

print(f"\n{'=' * 50}")
print("Download completed!")
print(f"Total files downloaded: {len(downloaded_files)}")

## Step 2: Download Raw Data from Kaggle

This adds an **English** deepfake/real voice dataset from Kaggle, so the final dataset is not only Indian languages but also has English fake and real samples.


In [ ]:
import kagglehub

kaggle_path = kagglehub.dataset_download("adarshsingh0903/audio-deepfake-detection-dataset")
print("Kaggle dataset downloaded to:", kaggle_path)

In [ ]:
# Check what folders came with the Kaggle dataset
!ls {kaggle_path}

## Step 3: Convert Parquet Files to WAV (Hugging Face data)

The Hugging Face datasets store audio as bytes inside parquet files. These two classes pull the audio bytes out and save them as normal `.wav` files, sorted into language folders.

**Disk space note:** parquet files are big, and a free Colab VM has limited storage. So both classes below delete each parquet file right after it is converted to WAV (`delete_after_process=True`, which is the default). Set it to `False` only if you want to keep the raw parquet files for some other reason.


### 3.1 IndicSynth -> WAV (goes into the `fake` folder)

In [ ]:
class IndicSynthToWAV:
    """Extracts audio bytes from IndicSynth parquet files and saves them as WAV."""

    def __init__(self, dataset_dir, dataset_out_dir):
        self.dataset_dir = dataset_dir

        self.languages = [
            directory for directory in os.listdir(self.dataset_dir)
            if os.path.isdir(os.path.join(self.dataset_dir, directory))
            and not directory.startswith('.')
        ]

        self.language_dirs = {
            lang: os.path.join(self.dataset_dir, lang)
            for lang in self.languages
        }

        self.dataset_out_dir = dataset_out_dir
        self.out_dirs = {
            lang: os.path.join(self.dataset_out_dir, lang)
            for lang in self.languages
        }

    def extract_audio(self, max_file_process=None, delete_after_process=True):
        # delete_after_process=True removes each parquet file right after it is
        # converted to WAV. Free Colab VMs have very little disk space, and these
        # parquet files are large, so keeping them around after use just wastes space.
        for lang in tqdm(self.languages, desc="Processing Languages"):
            lang_dir = self.language_dirs[lang]

            parquet_files = [
                file for file in os.listdir(lang_dir)
                if file.endswith(".parquet")
            ]

            # Calculate limit separately for each language
            file_limit = (
                len(parquet_files)
                if max_file_process is None
                else min(max_file_process, len(parquet_files))
            )

            processed_count = 0

            with tqdm(total=file_limit, desc=f"{lang} Parquet Files", leave=False) as file_pbar:
                for file in parquet_files:
                    if processed_count >= file_limit:
                        break

                    file_path = os.path.join(lang_dir, file)

                    # Skip empty files
                    if os.path.getsize(file_path) == 0:
                        print(f"\nSkipping empty file: {file}")
                        continue

                    try:
                        df = pd.read_parquet(file_path)

                        # Remove duplicate audio
                        df = df.drop_duplicates(subset=['Source Reference Audio'])
                        df = df.reset_index(drop=True)

                        self.extract_audio_bytes(df, self.out_dirs[lang], processed_count, lang)

                        processed_count += 1
                        file_pbar.update(1)

                        # File is fully processed now, safe to delete it and free disk space
                        if delete_after_process:
                            freed_mb = os.path.getsize(file_path) / (1024 * 1024)
                            os.remove(file_path)
                            print(f"Deleted (freed {freed_mb:.1f} MB): {file_path}")

                    except Exception as e:
                        print(f"\nError reading {file}: {e}. Skipping file.")
                        continue

    def extract_audio_bytes(self, df, out_dir, curr_idx, lang):
        os.makedirs(out_dir, exist_ok=True)

        for df_idx, row in tqdm(
            df.iterrows(), total=len(df),
            desc=f"{lang} File {curr_idx + 1} Audio", leave=False
        ):
            try:
                audio_bytes = row['audio']['bytes']

                output_filename = os.path.join(
                    out_dir,
                    f"{curr_idx}_{df_idx}_{row['Generative Model']}_{row['Gender']}.wav"
                )

                with open(output_filename, 'wb') as f:
                    f.write(audio_bytes)

            except Exception as e:
                print(f"\nError extracting row {df_idx}: {e}")

In [ ]:
indic = IndicSynthToWAV(dataset_dir=INDIC_SYNTH_RAW_DIR, dataset_out_dir=FAKE_DIR)

# Uncomment to actually run the extraction (kept commented so re-running the notebook
# top to bottom doesn't redo a long extraction by accident)
# delete_after_process=True (default) deletes each parquet file as soon as it is converted
# indic.extract_audio(max_file_process=None, delete_after_process=True)

### 3.2 IndicVoices -> WAV (goes into the `real` folder)

In [ ]:
class IndicVoicesToWAV:
    """Extracts audio bytes from IndicVoices parquet files and saves them as WAV."""

    def __init__(self, dataset_dir, dataset_out_dir):
        self.dataset_dir = dataset_dir

        self.languages = [
            directory for directory in os.listdir(self.dataset_dir)
            if os.path.isdir(os.path.join(self.dataset_dir, directory))
            and not directory.startswith('.')
        ]

        self.language_dirs = {
            lang: os.path.join(self.dataset_dir, lang)
            for lang in self.languages
        }

        self.dataset_out_dir = dataset_out_dir
        self.out_dirs = {
            lang: os.path.join(self.dataset_out_dir, lang.capitalize())
            for lang in self.languages
        }

    def extract_audio(self, max_file_process=None, delete_after_process=True):
        # delete_after_process=True removes each parquet file right after it is
        # converted to WAV, since a free Colab VM does not have much disk space.
        for lang in tqdm(self.languages, desc="Processing Languages"):
            lang_dir = self.language_dirs[lang]

            parquet_files = [
                file for file in os.listdir(lang_dir)
                if file.endswith(".parquet")
            ]

            file_limit = (
                len(parquet_files)
                if max_file_process is None
                else min(max_file_process, len(parquet_files))
            )

            processed_count = 0

            with tqdm(total=file_limit, desc=f"{lang} Parquet Files", leave=False) as file_pbar:
                for file in parquet_files:
                    if processed_count >= file_limit:
                        break

                    file_path = os.path.join(lang_dir, file)

                    if os.path.getsize(file_path) == 0:
                        print(f"\nSkipping empty file: {file}")
                        continue

                    try:
                        df = pd.read_parquet(file_path)

                        # Random sample so we don't take every single row
                        df = df.sample(n=min(200, len(df)))
                        df = df.reset_index(drop=True)

                        self.extract_audio_bytes(df, self.out_dirs[lang], processed_count, lang)

                        processed_count += 1
                        file_pbar.update(1)

                        # File is fully processed now, safe to delete it and free disk space
                        if delete_after_process:
                            freed_mb = os.path.getsize(file_path) / (1024 * 1024)
                            os.remove(file_path)
                            print(f"Deleted (freed {freed_mb:.1f} MB): {file_path}")

                    except Exception as e:
                        print(f"\nError reading {file}: {e}. Skipping file.")
                        continue

    def extract_audio_bytes(self, df, out_dir, curr_idx, lang):
        os.makedirs(out_dir, exist_ok=True)

        for df_idx, row in tqdm(
            df.iterrows(), total=len(df),
            desc=f"{lang} File {curr_idx + 1} Audio", leave=False
        ):
            try:
                audio_bytes = row['audio_filepath']['bytes']

                output_filename = os.path.join(
                    out_dir,
                    f"{curr_idx}_{df_idx}_{row['samples']}_{row['gender']}.wav"
                )

                with open(output_filename, 'wb') as f:
                    f.write(audio_bytes)

            except Exception as e:
                print(f"\nError extracting row {df_idx}: {e}")

In [ ]:
indic_v = IndicVoicesToWAV(dataset_dir=INDIC_VOICES_RAW_DIR, dataset_out_dir=REAL_DIR)

# delete_after_process=True (default) deletes each parquet file as soon as it is converted
indic_v.extract_audio(max_file_process=None, delete_after_process=True)

## Step 4: Organize the Kaggle Dataset (English)

The Kaggle dataset has real English samples and several fake/synthetic English TTS model folders. We copy each into the right place inside `real/English` or `fake/English`, renaming files so nothing overwrites anything else.


### 4.1 Real English samples

In [ ]:
real_english_dir = os.path.join(REAL_DIR, "English")
os.makedirs(real_english_dir, exist_ok=True)

source_dir = os.path.join(kaggle_path, "real_samples")

if os.path.exists(source_dir):
    print("Copying real English samples...")
    shutil.copytree(source_dir, real_english_dir, dirs_exist_ok=True)
    print(f"Successfully copied to: {real_english_dir}")
else:
    print(f"Error: Source path {source_dir} not found.")

### 4.2 Fake English samples (multiple TTS models)

In [ ]:
fake_english_dir = os.path.join(FAKE_DIR, "English")
os.makedirs(fake_english_dir, exist_ok=True)

# Every one of these folders holds WAV files from a different fake-voice generator
tts_source_folders = [
    "VoiceBox",
    "VALLE",
    "FlashSpeech",
    "NaturalSpeech3",
    "PromptTTS2",
    "seedtts_files",
    "xTTS"
]

for folder_name in tts_source_folders:
    source_dir = os.path.join(kaggle_path, folder_name)

    if not os.path.exists(source_dir):
        print(f"Not found: {source_dir}")
        continue

    wav_files = sorted(glob.glob(os.path.join(source_dir, "*.wav")))

    print(f"\nProcessing: {folder_name}")
    print(f"   Found {len(wav_files)} WAV files")

    for index, old_path in enumerate(wav_files, start=1):
        # Example rename: 1_VoiceBox.wav, 2_VoiceBox.wav ...
        new_name = f"{index}_{folder_name}.wav"
        new_path = os.path.join(fake_english_dir, new_name)

        # Copy without touching the original Kaggle files
        shutil.copy2(old_path, new_path)

    print(f"   Copied {len(wav_files)} files")

print("\n" + "=" * 50)
print("Fake English (multi-model) processing done")
print(f"Stored in: {fake_english_dir}")
print("=" * 50)

### 4.3 Fake English samples (OpenAI voices, kept separate because filenames need special parsing)

In [ ]:
source_dir = os.path.join(kaggle_path, "OpenAI")

wav_files = glob.glob(os.path.join(source_dir, "*.wav"))

# Sort naturally so alloy_0, alloy_1, ... alloy_10 stay in the right order
# (plain sort would put alloy_10 before alloy_2)
def natural_sort_key(path):
    filename = os.path.basename(path)
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", filename)
    ]

wav_files = sorted(wav_files, key=natural_sort_key)

global_id = 1
for old_path in wav_files:
    old_filename = os.path.basename(old_path)
    filename_without_ext = os.path.splitext(old_filename)[0]

    # Filenames look like: alloy_0, echo_25, fable_17
    # voice_name = the part before "_<number>"
    voice_name = re.sub(r"_\d+$", "", filename_without_ext)

    match = re.search(r"_(\d+)$", filename_without_ext)
    original_id = match.group(1) if match else "unknown"

    new_name = f"{global_id}_OpenAI_{voice_name}_{original_id}.wav"
    new_path = os.path.join(fake_english_dir, new_name)

    shutil.copy2(old_path, new_path)
    print(f"Copied: {old_filename} -> {new_name}")

    global_id += 1

print("\n" + "=" * 60)
print("OpenAI dataset processing completed")
print(f"Source      : {source_dir}")
print(f"Destination : {fake_english_dir}")
print(f"Files copied: {len(wav_files)}")
print("=" * 60)

## Step 5: Preview Some Sample Audio Files

Quick sanity check: play one fake sample and one real sample straight from the final dataset folder, to confirm the files actually saved correctly.


In [ ]:
import IPython.display as ipd

def show_sample_audio(class_dir, class_label, num_samples=2):
    """Finds a few WAV files under class_dir and plays them inline."""
    sample_files = glob.glob(os.path.join(class_dir, "**", "*.wav"), recursive=True)[:num_samples]

    if not sample_files:
        print(f"No WAV files found yet under: {class_dir}")
        return

    for f in sample_files:
        print(f"{class_label}: {f}")
        ipd.display(ipd.Audio(f))

show_sample_audio(FAKE_DIR, "FAKE sample")
show_sample_audio(REAL_DIR, "REAL sample")

## Step 6: Check Dataset Size

Before archiving, check how big the final dataset folder is.


In [ ]:
!du -sh "{DATASET_ROOT}"
!du -sh "{FAKE_DIR}"
!du -sh "{REAL_DIR}"

## Step 7: Create the Final Archive

Package the whole `indian-deepfake-voice` folder into one `.tar.gz` file, so it is easy to move, share, or upload anywhere.


In [ ]:
%cd /content

In [ ]:
!tar -czf indian-deepfake-voice.tar.gz indian-deepfake-voice

In [ ]:
ARCHIVE_PATH = "/content/indian-deepfake-voice.tar.gz"
!du -sh "{ARCHIVE_PATH}"

## Step 8: Backup the Archive to Google Drive

Drive is only mounted here, at the very end, just to copy the single `.tar.gz` file for permanent storage. This avoids the slowness/rate limits of working on Drive directly during download and processing.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

In [ ]:
drive_archive_path = os.path.join(DRIVE_BACKUP_DIR, "indian-deepfake-voice.tar.gz")

shutil.copy2(ARCHIVE_PATH, drive_archive_path)

print("Backup complete!")
print(f"Local archive : {ARCHIVE_PATH}")
print(f"Drive backup  : {drive_archive_path}")

In [ ]:
print("Local archive size:")
!du -sh "{ARCHIVE_PATH}"

print("\nDrive backup size:")
!du -sh "{drive_archive_path}"

## Step 9: Cleanup (optional)

Step 3 already deletes each parquet file right after it turns into WAV, so `INDIC_SYNTH_RAW_DIR` / `INDIC_VOICES_RAW_DIR` should already be mostly empty. This step is a backup, for extra space, or if you ever ran extraction with `delete_after_process=False`. Kept commented so nothing is deleted by accident.


In [ ]:
# Remove any leftover raw parquet folders
# !rm -rf "{INDIC_SYNTH_RAW_DIR}"
# !rm -rf "{INDIC_VOICES_RAW_DIR}"

# Remove the uncompressed dataset folder (only after confirming the .tar.gz is good)
# !rm -rf "{DATASET_ROOT}"